# 02 - Preprocessing V3 (Weather + True t+1 Forecasting)

This notebook merges daily Open-Meteo weather features into the AQI pipeline and rebuilds true next-day forecasting splits.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

POLLUTION_PATH = Path('data/processed/combined_all_cities.csv')
WEATHER_PATH = Path('data/raw/weather_all_cities.csv')
SPLITS_DIR = Path('data/splits')
MODELS_DIR = Path('outputs/models')

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

POLLUTANT_COLUMNS = ['PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3']
WEATHER_COLUMNS = [
    'temperature_2m_max',
    'temperature_2m_min',
    'precipitation_sum',
    'wind_speed_10m_max',
    'relative_humidity_2m_max',
]
WEATHER_LAG_COLUMNS = ['wind_speed_lag1', 'precip_lag1', 'temp_max_lag1', 'humidity_lag1']

BASE_FEATURES = [
    'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3',
    'PM2.5_lag1', 'PM2.5_lag7', 'PM2.5_lag30',
    'AQI_lag1', 'AQI_lag7',
    'PM2.5_roll7_mean', 'PM2.5_roll7_std', 'PM2.5_roll30_mean',
    'day_of_week', 'month', 'season', 'is_weekend', 'year',
]
FEATURE_COLUMNS = BASE_FEATURES + WEATHER_COLUMNS + WEATHER_LAG_COLUMNS

POLLUTION_REQUIRED = ['Timestamp', 'City', 'Location', 'AQI'] + POLLUTANT_COLUMNS
WEATHER_REQUIRED = ['Timestamp', 'City'] + WEATHER_COLUMNS


## Step 1 - Load And Validate

In [ ]:
if not POLLUTION_PATH.exists():
    raise FileNotFoundError(f'Missing pollution input: {POLLUTION_PATH}')
if not WEATHER_PATH.exists():
    raise FileNotFoundError(f'Missing weather input: {WEATHER_PATH}')

pollution = pd.read_csv(POLLUTION_PATH)
weather = pd.read_csv(WEATHER_PATH)

pollution['Timestamp'] = pd.to_datetime(pollution['Timestamp'], errors='coerce')
weather['Timestamp'] = pd.to_datetime(weather['Timestamp'], errors='coerce')

missing_pollution = [c for c in POLLUTION_REQUIRED if c not in pollution.columns]
missing_weather = [c for c in WEATHER_REQUIRED if c not in weather.columns]
if missing_pollution:
    raise ValueError(f'Missing pollution columns: {missing_pollution}')
if missing_weather:
    raise ValueError(f'Missing weather columns: {missing_weather}')

pollution = pollution.dropna(subset=['Timestamp']).copy()
weather = weather.dropna(subset=['Timestamp']).copy()

for col in POLLUTANT_COLUMNS + ['AQI']:
    pollution[col] = pd.to_numeric(pollution[col], errors='coerce')
for col in WEATHER_COLUMNS:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

print('Pollution shape:', pollution.shape)
print('Weather shape:', weather.shape)
print('Pollution columns:', pollution.columns.tolist())
print('Weather columns:', weather.columns.tolist())


## Step 2 - Merge

In [ ]:
before_shape = pollution.shape
merged = pollution.merge(weather[WEATHER_REQUIRED], on=['Timestamp', 'City'], how='left')

print('Shape before merge:', before_shape)
print('Shape after merge :', merged.shape)
print('Weather null counts after merge:')
print(merged[WEATHER_COLUMNS].isna().sum())


## Step 3 - Handle Weather Nulls

In [ ]:
merged = merged.sort_values(['City', 'Timestamp']).reset_index(drop=True)
merged[WEATHER_COLUMNS] = merged.groupby('City', group_keys=False)[WEATHER_COLUMNS].ffill(limit=3)

before_weather_drop = len(merged)
merged = merged.dropna(subset=WEATHER_COLUMNS).copy()
print('Dropped rows with remaining weather nulls:', before_weather_drop - len(merged))
print('Weather null counts after handling:')
print(merged[WEATHER_COLUMNS].isna().sum())


## Steps 4-5 - Drop Helpers And Sort

In [ ]:
if 'Month' in merged.columns:
    merged = merged.drop(columns=['Month'])
    print('Dropped old Month column.')

merged = merged.sort_values(['City', 'Timestamp']).reset_index(drop=True)
print('Sorted shape:', merged.shape)


## Steps 6-8 - Feature Engineering, Weather Lags, Target

In [ ]:
# Causal fill for pollution fields. No backward fill.
merged[POLLUTANT_COLUMNS] = merged.groupby('City', group_keys=False)[POLLUTANT_COLUMNS].ffill(limit=3)
merged['AQI'] = merged.groupby('City', group_keys=False)['AQI'].ffill(limit=3)
merged = merged.dropna(subset=['PM2.5', 'AQI']).copy()

grp = merged.groupby('City', group_keys=False)

merged['PM2.5_lag1'] = grp['PM2.5'].shift(1)
merged['PM2.5_lag7'] = grp['PM2.5'].shift(7)
merged['PM2.5_lag30'] = grp['PM2.5'].shift(30)
merged['AQI_lag1'] = grp['AQI'].shift(1)
merged['AQI_lag7'] = grp['AQI'].shift(7)

# Rolling windows are causal at time t: they use observations up to and including t.
merged['PM2.5_roll7_mean'] = grp['PM2.5'].transform(lambda s: s.rolling(7, min_periods=7).mean())
merged['PM2.5_roll7_std'] = grp['PM2.5'].transform(lambda s: s.rolling(7, min_periods=7).std())
merged['PM2.5_roll30_mean'] = grp['PM2.5'].transform(lambda s: s.rolling(30, min_periods=30).mean())

merged['day_of_week'] = merged['Timestamp'].dt.dayofweek
merged['month'] = merged['Timestamp'].dt.month
merged['year'] = merged['Timestamp'].dt.year
merged['is_weekend'] = (merged['day_of_week'] >= 5).astype(int)

def month_to_season(month):
    if month in (12, 1, 2):
        return 0
    if month in (3, 4, 5):
        return 1
    if month in (6, 7, 8):
        return 2
    return 3

merged['season'] = merged['month'].apply(month_to_season).astype(int)

merged['wind_speed_lag1'] = grp['wind_speed_10m_max'].shift(1)
merged['precip_lag1'] = grp['precipitation_sum'].shift(1)
merged['temp_max_lag1'] = grp['temperature_2m_max'].shift(1)
merged['humidity_lag1'] = grp['relative_humidity_2m_max'].shift(1)

merged['AQI_t_plus_1'] = grp['AQI'].shift(-1)
merged['target_timestamp'] = grp['Timestamp'].shift(-1)

required_engineered = [
    'AQI_t_plus_1', 'target_timestamp',
    'PM2.5_lag1', 'PM2.5_lag7', 'PM2.5_lag30', 'AQI_lag1', 'AQI_lag7',
    'PM2.5_roll7_mean', 'PM2.5_roll7_std', 'PM2.5_roll30_mean',
] + WEATHER_LAG_COLUMNS

before_drop = len(merged)
merged = merged.dropna(subset=required_engineered).copy()
print('Dropped rows after feature/target engineering:', before_drop - len(merged))
print('Engineered shape:', merged.shape)


## Step 9 - Train/Test Split

In [ ]:
train = merged[merged['target_timestamp'] <= pd.Timestamp('2023-12-31')].copy()
test = merged[merged['target_timestamp'] >= pd.Timestamp('2024-01-01')].copy()

print('Train rows:', len(train))
print('Test rows :', len(test))
print('Train feature date range:', train['Timestamp'].min(), 'to', train['Timestamp'].max())
print('Train target date range :', train['target_timestamp'].min(), 'to', train['target_timestamp'].max())
print('Test feature date range :', test['Timestamp'].min(), 'to', test['Timestamp'].max())
print('Test target date range  :', test['target_timestamp'].min(), 'to', test['target_timestamp'].max())


## Steps 10-12 - Feature Set, Scaling, Save Outputs

In [ ]:
missing_features = [col for col in FEATURE_COLUMNS if col not in train.columns]
if missing_features:
    raise ValueError(f'Missing features: {missing_features}')

# Remaining feature NaNs are filled with train medians only.
train_features = train[FEATURE_COLUMNS].astype(float)
test_features = test[FEATURE_COLUMNS].astype(float)
train_medians = train_features.median(numeric_only=True)
print('Feature NaNs before median fill - train:', int(train_features.isna().sum().sum()))
print('Feature NaNs before median fill - test :', int(test_features.isna().sum().sum()))
train_features = train_features.fillna(train_medians)
test_features = test_features.fillna(train_medians)

scaler = StandardScaler()
train_scaled = pd.DataFrame(scaler.fit_transform(train_features), columns=FEATURE_COLUMNS, index=train.index)
test_scaled = pd.DataFrame(scaler.transform(test_features), columns=FEATURE_COLUMNS, index=test.index)

for col in FEATURE_COLUMNS:
    train[col] = train_scaled[col]
    test[col] = test_scaled[col]

joblib.dump(scaler, MODELS_DIR / 'scaler_v3.pkl')
(MODELS_DIR / 'feature_columns_v3.json').write_text(json.dumps(FEATURE_COLUMNS, indent=2), encoding='utf-8')

export_columns = ['Timestamp', 'target_timestamp', 'City', 'Location', 'AQI_t_plus_1'] + FEATURE_COLUMNS
train_out = train[export_columns].copy().rename(columns={'AQI_t_plus_1': 'AQI'})
test_out = test[export_columns].copy().rename(columns={'AQI_t_plus_1': 'AQI'})

train_out.to_csv(SPLITS_DIR / 'train_v3.csv', index=False)
test_out.to_csv(SPLITS_DIR / 'test_v3.csv', index=False)

print('Saved:', SPLITS_DIR / 'train_v3.csv')
print('Saved:', SPLITS_DIR / 'test_v3.csv')
print('Saved:', MODELS_DIR / 'scaler_v3.pkl')
print('Saved:', MODELS_DIR / 'feature_columns_v3.json')


## Step 13 - Assertions

In [ ]:
assert train_out[FEATURE_COLUMNS + ['AQI']].isna().sum().sum() == 0, 'Nulls found in train features/target.'
assert test_out[FEATURE_COLUMNS + ['AQI']].isna().sum().sum() == 0, 'Nulls found in test features/target.'
assert train_out['target_timestamp'].max() < test_out['target_timestamp'].min(), 'Train/test target date overlap.'
assert (pd.to_datetime(train_out['Timestamp']) < pd.to_datetime(train_out['target_timestamp'])).all(), 'Train timestamp leakage.'
assert (pd.to_datetime(test_out['Timestamp']) < pd.to_datetime(test_out['target_timestamp'])).all(), 'Test timestamp leakage.'

print('Assertions passed.')


## Step 14 - Summary

In [ ]:
print('Final train shape:', train_out.shape)
print('Final test shape :', test_out.shape)
print('Feature count:', len(FEATURE_COLUMNS))
print('Train sample rows:')
display(train_out.head())
print('Test sample rows:')
display(test_out.head())
